# web_agent — GOLD dataset run (Kaggle)

Real, leak-safe gold data (before+after images + task only). Separate from the synthetic notebook — the synthetic 70k pipeline is untouched. Heads with no gold labels (confidence/memory/recovery_success) are disabled in `qwen2vl_2b_gold.yaml`. Judge by **outcome_mcc** on the gold TEST split.

In [ ]:
# 1. Clone the package + install deps, make it importable (re-run safe)
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
SRC = f"{ROOT}/src"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}
!pip install -q -U "transformers>=4.49" peft bitsandbytes accelerate scikit-learn
for m in [k for k in list(sys.modules) if k == "web_agent" or k.startswith("web_agent.")]:
    del sys.modules[m]
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import web_agent
print("web_agent ready ->", list(web_agent.__path__))

In [ ]:
# 2. GPU + gold data path (auto-detected, slug-proof)
import os, glob
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))
hits = glob.glob("/kaggle/input/**/split_train.json", recursive=True)
assert hits, "WebGoldData not attached: right panel -> Add Input -> WebGoldData"
GOLD_PATH = os.path.dirname(hits[0])
print("GOLD_PATH =", GOLD_PATH)
print("splits:", [f for f in os.listdir(GOLD_PATH) if f.endswith(".json")])

In [ ]:
# 3. Config (gold) + processor + one gold batch
import torch
from transformers import AutoProcessor
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split

set_seed(42)
cfg = load_config("configs/backbones/qwen2vl_2b_gold.yaml")
cfg["data"]["root"] = GOLD_PATH
cfg["data"]["num_workers"] = 0

bb = cfg["backbone"]
processor = AutoProcessor.from_pretrained(
    bb["vlm_model"], min_pixels=bb["min_pixels"], max_pixels=bb["max_pixels"])

train = load_gold_split(cfg, "train")
print("gold train rows:", len(train))
loader = build_gold_dataloader(cfg, "train", processor, records=train,
                               limit=8, batch_size=4, num_workers=0)
batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:22} {tuple(v.shape)}  {v.dtype}")
print("outcome labels in batch:", batch["label_outcome"].tolist())

In [ ]:
# 4. Build model + gold class-weighted loss (3 heads disabled in the gold config)
from web_agent.models.model import WebAgentModel
from web_agent.models.loss import CombinedLoss
from web_agent.data.gold_dataloader import gold_class_weights

model = WebAgentModel(cfg)
print("VLM hidden dim D =", model.encoder.hidden_dim, "| pooling =", cfg["backbone"]["pooling"])
device = "cuda"
for m in (model.adapter, model.failure_head, model.action_head,
          model.memory_head, model.recovery_outcome_head):
    m.to(device)

aw, fw, ow = gold_class_weights(train)
print("action w :", [round(x,2) for x in aw.tolist()])
print("failtype w:", [round(x,2) for x in fw.tolist()])
print("outcome w:", [round(x,2) for x in ow.tolist()], "(capped)")
loss_fn = CombinedLoss(cfg, action_class_weights=aw.to(device),
                       failtype_class_weights=fw.to(device),
                       outcome_class_weights=ow.to(device)).to(device)
print("disabled heads -> confidence/memory_flag/recovery_outcome weights:",
      cfg["loss"]["confidence"], cfg["loss"]["memory_flag"], cfg["loss"]["recovery_outcome"])
print("trainable params:", f"{sum(p.numel() for p in model.trainable_parameters()):,}")

In [ ]:
# 5. SMOKE — one batch, loss finite, disabled heads contribute 0
model.train()
b = next(iter(loader))
with torch.autocast("cuda", dtype=torch.float16):
    preds = model(b)
    terms = loss_fn(preds, {k: (v.to(device) if torch.is_tensor(v) else v)
                            for k, v in b.items()})
print("loss terms:", {k: round(float(v.detach()), 4) for k, v in terms.items()})
assert torch.isfinite(terms["total"]), "loss NaN/inf - STOP"
for t, w in (("confidence","confidence"),("memory","memory_flag"),("recovery_outcome","recovery_outcome")):
    assert float(terms[t].detach()) * cfg["loss"][w] == 0.0, f"{t} not disabled"
print("contrastive non-zero:", float(terms["contrastive"].detach()) != 0.0)
print("peak GPU (GB):", round(torch.cuda.max_memory_allocated()/1e9, 2))
print("SMOKE PASS")

In [ ]:
# 6. TRAIN on gold + final eval on the gold TEST split
from web_agent.train.trainer import Trainer, collect_predictions, compute_metrics

cfg["train"]["epochs"] = 5
cfg["data"]["num_workers"] = 4
train_loader = build_gold_dataloader(cfg, "train", processor, shuffle=True, num_workers=4)
val_loader   = build_gold_dataloader(cfg, "val",   processor, shuffle=False, num_workers=4)

trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader, train_sampler=None)
print("training done:", trainer.fit())

test_loader = build_gold_dataloader(cfg, "test", processor, shuffle=False, num_workers=4)
m = compute_metrics(collect_predictions(model, test_loader, device))
print("GOLD TEST:", {k: round(v,4) for k,v in m.items()})
print("  HEADLINE = outcome_mcc / outcome_bal_acc / failure_macro_f1")
print("  (ignore memory_acc / recovery_outcome_acc / ece -- heads disabled on gold)")